In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [4]:
df = pd.read_csv("../data/processed/loan_feature_engineered.csv")

print("Shape:", df.shape)

Shape: (1303638, 93)


In [5]:
X = df.drop(columns=["default"])
y = df["default"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1303638, 92)
y shape: (1303638,)


In [6]:
print(y.value_counts())
print()
print(y.value_counts(normalize=True) * 100)

default
0    1041952
1     261686
Name: count, dtype: int64

default
0    79.926483
1    20.073517
Name: proportion, dtype: float64


Train/Test Split

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [8]:
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (1042910, 92)
X_test : (260728, 92)
y_train: (1042910,)
y_test : (260728,)


In [9]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [10]:
import joblib

joblib.dump(
    scaler,
    "../models/scaler.pkl"
)

['../models/scaler.pkl']

SMOTE - (Synthetic Minority Over-sampling Technique.)

In [11]:
pip install imbalanced-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
from imblearn.over_sampling import SMOTE

In [13]:
smote = SMOTE(
    random_state=42
)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train_scaled,
    y_train
)

In [14]:
print("Before SMOTE:")
print(y_train.value_counts())

print("\nAfter SMOTE:")
print(y_train_smote.value_counts())

Before SMOTE:
default
0    833561
1    209349
Name: count, dtype: int64

After SMOTE:
default
1    833561
0    833561
Name: count, dtype: int64


In [15]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logistic_model.fit(
    X_train_smote,
    y_train_smote
)

print("Logistic Regression trained successfully!")

Logistic Regression trained successfully!


In [16]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logistic_model.fit(
    X_train_smote,
    y_train_smote
)

print("Logistic Regression trained successfully!")

Logistic Regression trained successfully!


In [17]:
joblib.dump(
    logistic_model,
    "../models/logistic.pkl"
)

['../models/logistic.pkl']

In [20]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=2,
    class_weight="balanced"
)

rf_model.fit(
    X_train_scaled,
    y_train
)

print("Random Forest trained successfully!")

Random Forest trained successfully!


In [21]:
joblib.dump(
    rf_model,
    "../models/random_forest.pkl"
)

print("Random Forest saved successfully!")

Random Forest saved successfully!


In [23]:
pip install xgboost

  Using cached xgboost-3.4.0-py3-none-win_amd64.whl.metadata (2.0 kB)
Using cached xgboost-3.4.0-py3-none-win_amd64.whl (48.9 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
import xgboost as xgb

print(xgb.__version__)

3.4.0


In [25]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=42,
    n_jobs=2
)

xgb_model.fit(
    X_train_scaled,
    y_train
)

print("XGBoost trained successfully!")

XGBoost trained successfully!


In [26]:
joblib.dump(
    xgb_model,
    "../models/xgboost.pkl"
)

print("XGBoost saved successfully!")

XGBoost saved successfully!


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score
)

def get_metrics(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    }


logistic_metrics = get_metrics(
    logistic_model,
    X_test_scaled,
    y_test
)

rf_metrics = get_metrics(
    rf_model,
    X_test_scaled,
    y_test
)

xgb_metrics = get_metrics(
    xgb_model,
    X_test_scaled,
    y_test
)

print("Logistic Regression:", logistic_metrics)
print("Random Forest:", rf_metrics)
print("XGBoost:", xgb_metrics)